### Импорты и пути

In [1]:
import pandas as pd
import sqlite3
import os

BASE_PATH = 'C:/Users/menta/tmdb-sql-powerbi'
DATA_PATH = f'{BASE_PATH}/data/'
DB_PATH   = f'{BASE_PATH}/data/tmdb.db'

print("Пути настроены")

Пути настроены


### Загрузка и базовая очистка

In [2]:
df = pd.read_csv(f'{DATA_PATH}TMDB_movie_dataset_v11.csv')

# Оставляем только нужные столбцы
cols = ['id', 'title', 'release_date', 'runtime', 'budget',
        'revenue', 'popularity', 'vote_average', 'vote_count',
        'original_language', 'status', 'genres']

df = df[cols].copy()

# Базовая очистка
df = df[df['vote_average'] > 0]
df = df[df['vote_count'] >= 10]
df = df.dropna(subset=['release_date', 'title'])

# Типы данных
df['budget']  = pd.to_numeric(df['budget'],  errors='coerce').fillna(0)
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').fillna(0)
df['runtime'] = pd.to_numeric(df['runtime'], errors='coerce').fillna(0)

# Даты
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year']  = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month

df = df.dropna(subset=['release_year'])
df['release_year'] = df['release_year'].astype(int)

print(f"Загружено: {len(df):,} фильмов")
df.head(3)

Загружено: 78,814 фильмов


,id,title,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,original_language,status,genres,release_year,release_month
0,27205,Inception,2010-07-15,148,160000000,825532764,83.952,8.364,34495,en,Released,"Action, Science Fiction, Adventure",2010,7
1,157336,Interstellar,2014-11-05,169,165000000,701729206,140.241,8.417,32571,en,Released,"Adventure, Drama, Science Fiction",2014,11
2,155,The Dark Knight,2008-07-16,152,185000000,1004558444,130.643,8.512,30619,en,Released,"Drama, Action, Crime, Thriller",2008,7


### Создание таблицы films

In [3]:
films = df[[
    'id', 'title', 'release_year', 'release_month',
    'runtime', 'budget', 'revenue', 'popularity',
    'vote_average', 'vote_count', 'original_language', 'status'
]].copy()

films = films.rename(columns={'id': 'film_id'})
films = films.drop_duplicates(subset='film_id')

print(f"Таблица films: {len(films):,} строк")
films.head(3)

Таблица films: 78,814 строк


,film_id,title,release_year,release_month,runtime,budget,revenue,popularity,vote_average,vote_count,original_language,status
0,27205,Inception,2010,7,148,160000000,825532764,83.952,8.364,34495,en,Released
1,157336,Interstellar,2014,11,169,165000000,701729206,140.241,8.417,32571,en,Released
2,155,The Dark Knight,2008,7,152,185000000,1004558444,130.643,8.512,30619,en,Released


### Создание таблицы film_genres

In [4]:
# Разбиваем строку жанров на отдельные строки
genres_df = df[['id', 'genres']].copy()
genres_df = genres_df.rename(columns={'id': 'film_id'})
genres_df['genres'] = genres_df['genres'].fillna('Unknown')

# Разбиваем 'Action, Drama' → две отдельные строки
film_genres = (
    genres_df
    .assign(genre=genres_df['genres'].str.split(','))
    .explode('genre')
)
film_genres['genre'] = film_genres['genre'].str.strip()
film_genres = film_genres[['film_id', 'genre']].drop_duplicates()
film_genres = film_genres[film_genres['genre'] != '']

print(f"Таблица film_genres: {len(film_genres):,} строк")
film_genres.head(5)

Таблица film_genres: 165,396 строк


,film_id,genre
0,27205,Action
0,27205,Science Fiction
0,27205,Adventure
1,157336,Adventure
1,157336,Drama


### Создание таблицы genre_stats

In [5]:
genre_stats = (
    film_genres[film_genres['genre'] != 'Unknown']
    .merge(films[['film_id', 'vote_average', 'runtime', 'budget']], on='film_id')
    .groupby('genre')
    .agg(
        avg_rating=('vote_average', 'mean'),
        avg_runtime=('runtime', 'mean'),
        avg_budget_mln=('budget', lambda x: round(x.mean() / 1e6, 1)),
        film_count=('film_id', 'nunique')
    )
    .round(2)
    .reset_index()
    .query('film_count >= 50')
    .sort_values('avg_rating', ascending=False)
)

print(f"Таблица genre_stats: {len(genre_stats):,} строк")
genre_stats.head(5)

Таблица genre_stats: 19 строк


,genre,avg_rating,avg_runtime,avg_budget_mln,film_count
5,Documentary,6.74,78.91,0.1,6178
11,Music,6.69,94.22,2.0,3114
9,History,6.60,114.86,5.5,2744
2,Animation,6.58,47.17,4.4,6349
17,War,6.47,107.15,5.0,2169


### Создание таблицы genres_combined

In [6]:
genres_combined = (
    film_genres[film_genres['genre'] != 'Unknown']
    .groupby('film_id')['genre']
    .apply(lambda x: ', '.join(sorted(x)))
    .reset_index()
    .rename(columns={'genre': 'genres_list'})
)

print(f"Таблица genres_combined: {len(genres_combined):,} строк")
genres_combined.head(5)

Таблица genres_combined: 77,934 строк


,film_id,genres_list
0,2,"Comedy, Drama, Romance"
1,3,"Comedy, Drama, Romance"
2,5,Comedy
3,6,"Action, Crime, Thriller"
4,8,Documentary


### Создание таблицы financial_metrics

In [7]:
financial = films[['film_id', 'budget', 'revenue']].copy()

financial = financial[
    (financial['budget'] > 0) &
    (financial['revenue'] > 0)
]

financial['profit'] = financial['revenue'] - financial['budget']
financial['roi']    = financial['profit'] / financial['budget']

# Убираем выбросы по ROI — оставляем до 99-го перцентиля
roi_99 = financial['roi'].quantile(0.99)
financial = financial[financial['roi'] <= roi_99]

financial = financial[['film_id', 'profit', 'roi']]

print(f"Таблица financial_metrics: {len(financial):,} строк")
print(f"Средний ROI: {financial['roi'].mean():.2f}")
print(f"Медианный ROI: {financial['roi'].median():.2f}")
financial.head(3)

Таблица financial_metrics: 8,825 строк
Средний ROI: 2.62
Медианный ROI: 0.82


,film_id,profit,roi
0,27205,665532764,4.159580
1,157336,536729206,3.252904
2,155,819558444,4.430046


### Создание таблицы roi_by_genre

In [8]:
roi_by_genre = (
    film_genres[film_genres['genre'] != 'Unknown']
    .merge(financial[['film_id', 'roi', 'profit']], on='film_id')
    .groupby('genre')
    .agg(
        avg_roi=('roi', 'mean'),
        avg_profit_mln=('profit', lambda x: round(x.mean() / 1e6, 1)),
        film_count=('film_id', 'nunique')
    )
    .round(2)
    .reset_index()
    .query('film_count >= 10')
    .sort_values('avg_roi', ascending=False)
)

print(f"Таблица roi_by_genre: {len(roi_by_genre):,} строк")
print(roi_by_genre.to_string(index=False))

Таблица roi_by_genre: 19 строк
          genre  avg_roi  avg_profit_mln  film_count
    Documentary     4.95            16.7          95
         Horror     4.12            28.6         999
          Music     3.21            32.4         319
      Animation     2.87           124.9         487
        Romance     2.80            34.7        1687
        Mystery     2.72            32.7         764
       TV Movie     2.60            13.6          12
         Comedy     2.59            44.0        3210
      Adventure     2.54           125.4        1520
       Thriller     2.50            40.6        2214
         Family     2.42            95.7         896
        Western     2.39            19.1         152
Science Fiction     2.39            97.4         930
            War     2.35            35.8         340
          Drama     2.35            28.3        4114
         Action     2.13            82.2        2192
        Fantasy     2.08           105.5         853
          Crime

### Запись в SQLite

In [9]:
conn = sqlite3.connect(DB_PATH)

films.to_sql('films', conn, if_exists='replace', index=False)
film_genres.to_sql('film_genres', conn, if_exists='replace', index=False)
financial.to_sql('financial_metrics', conn, if_exists='replace', index=False)
genre_stats.to_sql('genre_stats', conn, if_exists='replace', index=False)
genres_combined.to_sql('genres_combined', conn, if_exists='replace', index=False)
roi_by_genre.to_sql('roi_by_genre', conn, if_exists='replace', index=False)

conn.close()

print(f"База данных создана: {DB_PATH}")
print(f"Размер файла: {os.path.getsize(DB_PATH) / 1024 / 1024:.1f} МБ")

База данных создана: C:/Users/menta/tmdb-sql-powerbi/data/tmdb.db
Размер файла: 10.7 МБ


### Проверка

In [10]:
conn = sqlite3.connect(DB_PATH)

for table in ['films', 'film_genres', 'financial_metrics', 'genre_stats', 'genres_combined', 'roi_by_genre']:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn).iloc[0,0]
    print(f"{table}: {count:,} строк")

# Тестовый запрос
test = pd.read_sql("""
    SELECT f.title, f.vote_average, f.release_year, g.genre
    FROM films f
    JOIN film_genres g ON f.film_id = g.film_id
    LIMIT 5
""", conn)

conn.close()
print()
print(test)

films: 78,814 строк
film_genres: 165,396 строк
financial_metrics: 8,825 строк
genre_stats: 19 строк
genres_combined: 77,934 строк
roi_by_genre: 19 строк

          title  vote_average  release_year            genre
0     Inception         8.364          2010           Action
1     Inception         8.364          2010        Adventure
2     Inception         8.364          2010  Science Fiction
3  Interstellar         8.417          2014        Adventure
4  Interstellar         8.417          2014            Drama


In [11]:
conn = sqlite3.connect(DB_PATH)
print("Подключение к БД установлено")

Подключение к БД установлено


### Запрос 1 — Общая статистика по жанрам

In [12]:
q1 = pd.read_sql("""
    SELECT
        g.genre,
        COUNT(DISTINCT f.film_id)          AS film_count,
        ROUND(AVG(f.vote_average), 2)      AS avg_rating,
        ROUND(AVG(f.runtime), 0)           AS avg_runtime_min,
        ROUND(AVG(f.budget) / 1e6, 1)     AS avg_budget_mln,
        ROUND(AVG(fm.roi), 2)              AS avg_roi
    FROM films f
    JOIN film_genres g  ON f.film_id = g.film_id
    JOIN financial_metrics fm ON f.film_id = fm.film_id
    WHERE f.vote_count >= 10
      AND f.budget > 1000000
      AND f.revenue > 0
      AND g.genre NOT IN ('Unknown', '')
    GROUP BY g.genre
    HAVING film_count >= 50
    ORDER BY avg_rating DESC
""", conn)

print(q1.to_string(index=False))

          genre  film_count  avg_rating  avg_runtime_min  avg_budget_mln  avg_roi
        History         449        6.88            133.0            28.7     1.24
            War         317        6.83            129.0            29.3     1.74
      Animation         465        6.74             90.0            56.4     2.70
    Documentary          55        6.71             93.0             7.7     3.58
          Drama        3629        6.65            118.0            21.6     1.86
        Western         142        6.59            118.0            23.7     1.74
          Music         277        6.57            111.0            19.3     2.33
          Crime        1249        6.46            112.0            26.3     1.57
        Fantasy         816        6.44            108.0            55.0     2.01
        Romance        1491        6.43            114.0            20.5     2.35
         Family         861        6.43             97.0            49.3     2.33
      Adventure 

### Запрос 2 — Динамика рынка по годам с оконной функцией

In [13]:
q2 = pd.read_sql("""
    SELECT
        release_year,
        COUNT(*)                              AS film_count,
        ROUND(AVG(vote_average), 2)           AS avg_rating,
        ROUND(AVG(AVG(vote_average)) OVER (
            ORDER BY release_year
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2)                                 AS moving_avg_3y,
        ROUND(SUM(budget) / 1e9, 1)          AS total_budget_bln
    FROM films
    WHERE release_year BETWEEN 1980 AND 2024
      AND vote_count >= 10
    GROUP BY release_year
    ORDER BY release_year
""", conn)

print(q2.to_string(index=False))

 release_year  film_count  avg_rating  moving_avg_3y  total_budget_bln
         1980         495        6.04           6.04               0.7
         1981         508        6.15           6.09               0.7
         1982         515        6.07           6.09               0.8
         1983         515        6.09           6.10               0.7
         1984         513        6.10           6.09               1.0
         1985         524        6.05           6.08               1.1
         1986         574        6.05           6.07               1.3
         1987         653        5.97           6.02               1.3
         1988         657        5.98           6.00               1.6
         1989         659        6.05           6.00               1.7
         1990         584        6.01           6.01               2.0
         1991         612        6.08           6.05               2.4
         1992         609        6.17           6.09               2.3
      

### Запрос 3 — Топ-3 фильма по ROI в каждом жанре

In [14]:
q3 = pd.read_sql("""
    SELECT * FROM (
        SELECT
            g.genre,
            f.title,
            f.release_year,
            ROUND(fm.roi, 1)              AS roi,
            ROUND(fm.profit / 1e6, 1)    AS profit_mln,
            f.vote_average,
            RANK() OVER (
                PARTITION BY g.genre
                ORDER BY fm.roi DESC
            ) AS rank_in_genre
        FROM films f
        JOIN film_genres g  ON f.film_id = g.film_id
        JOIN financial_metrics fm ON f.film_id = fm.film_id
        WHERE f.budget > 1000000
          AND g.genre NOT IN ('Unknown', '')
    ) ranked
    WHERE rank_in_genre <= 3
    ORDER BY genre, rank_in_genre
""", conn)

print(q3.to_string(index=False))

          genre                          title  release_year  roi  profit_mln  vote_average  rank_in_genre
         Action                      Star Wars          1977 69.5       764.4         8.204              1
         Action                Vanishing Point          1971 54.6        71.0         7.182              2
         Action                     Goldfinger          1964 49.0       122.4         7.345              3
      Adventure                      Star Wars          1977 69.5       764.4         8.204              1
      Adventure                           Jaws          1975 66.2       463.7         7.659              2
      Adventure               Crocodile Dundee          1986 64.6       323.2         6.385              3
      Animation                   The Rescuers          1977 58.3        70.0         6.752              1
      Animation                The Jungle Book          1967 50.5       201.8         7.276              2
      Animation                      

### Запрос 4 — Сегментация фильмов

In [15]:
q4 = pd.read_sql("""
    SELECT
        CASE
            WHEN budget > 100000000 AND vote_average >= 7 THEN 'Успешный блокбастер'
            WHEN budget > 100000000 AND vote_average < 7  THEN 'Дорогой провал'
            WHEN budget BETWEEN 1000000 AND 100000000
                 AND vote_average >= 7                    THEN 'Инди-жемчужина'
            WHEN budget BETWEEN 1000000 AND 100000000
                 AND vote_average < 7                     THEN 'Обычный фильм'
            ELSE 'Без данных о бюджете'
        END                                AS segment,
        COUNT(*)                           AS film_count,
        ROUND(AVG(vote_average), 2)        AS avg_rating,
        ROUND(AVG(CASE WHEN f.budget > 0 AND f.revenue > 0
                       THEN fm.roi END), 2) AS avg_roi,
        ROUND(AVG(fm.profit) / 1e6, 1)    AS avg_profit_mln
    FROM films f
    JOIN financial_metrics fm ON f.film_id = fm.film_id
    WHERE vote_count >= 50
      AND segment != 'Без данных о бюджете'
    GROUP BY segment
    ORDER BY avg_rating DESC
""", conn)

print(q4.to_string(index=False))

            segment  film_count  avg_rating  avg_roi  avg_profit_mln
Успешный блокбастер         159        7.50     3.10           548.4
     Инди-жемчужина        1822        7.44     4.26            64.6
     Дорогой провал         284        6.31     1.63           250.8
      Обычный фильм        5033        6.11     1.71            28.5


### Запрос 5 — Фильмы выше среднего рейтинга своего жанра

In [16]:
q5 = pd.read_sql("""
    SELECT
        f.title,
        f.release_year,
        g.genre,
        f.vote_average,
        ROUND(genre_avg.avg_rating, 2)     AS genre_avg_rating,
        ROUND(f.vote_average
              - genre_avg.avg_rating, 2)   AS above_avg_by
    FROM films f
    JOIN film_genres g ON f.film_id = g.film_id
    JOIN (
        SELECT
            fg.genre,
            AVG(fl.vote_average) AS avg_rating
        FROM film_genres fg
        JOIN films fl ON fg.film_id = fl.film_id
        WHERE fl.vote_count >= 10
        GROUP BY fg.genre
    ) genre_avg ON g.genre = genre_avg.genre
    WHERE f.vote_average > genre_avg.avg_rating
      AND f.vote_count >= 100
      AND g.genre NOT IN ('Unknown', '')
    ORDER BY above_avg_by DESC
    LIMIT 20
""", conn)

print(q5.to_string(index=False))

                                             title  release_year           genre  vote_average  genre_avg_rating  above_avg_by
                       La Leyenda de los Chaneques          2023          Horror         8.452              5.35          3.10
                                            Psycho          1960          Horror         8.437              5.35          3.09
                 Franco Escamilla: por la anécdota          2018          Comedy         9.000              6.10          2.90
                                       The Shining          1980          Horror         8.218              5.35          2.87
                        Michael Jackson's Thriller          1983          Horror         8.163              5.35          2.82
Abraham Lincoln Vampire Hunter: The Great Calamity          2012          Horror         8.153              5.35          2.81
                                             Alien          1979          Horror         8.147              5.3

In [17]:
conn.close()
print("Соединение закрыто")

Соединение закрыто
